In [1]:
!pip install transformers torch biopython

In [2]:
import random
from Bio import SeqIO

positive = list(SeqIO.parse("uniprotkb_petase_2026_06_12.fasta", "fasta"))
print(f"Позитивных: {len(positive)}")

negative_all = list(SeqIO.parse("uniprotkb_reviewed_true_NOT_petase_NOT_2026_06_12.fasta", "fasta"))
print(f"Негативных всего: {len(negative_all)}")

random.seed(42)
negative = random.sample(negative_all, 5000)
print(f"Негативных после обрезки: {len(negative)}")

Позитивных: 4212
Негативных всего: 291011
Негативных после обрезки: 5000


In [3]:
sequences = []
labels = []

for record in positive:
    sequences.append(str(record.seq))
    labels.append(1)

for record in negative:
    sequences.append(str(record.seq))
    labels.append(0)

print(f"Всего примеров: {len(sequences)}")

Всего примеров: 9212


In [4]:
import torch
from transformers import AutoTokenizer, AutoModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используем: {device}")

tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t12_35M_UR50D")
model = AutoModel.from_pretrained("facebook/esm2_t12_35M_UR50D")
model = model.to(device)
model.eval()

print("Модель загружена")

Используем: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/778 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/136M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t12_35M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Модель загружена


In [5]:
import numpy as np
from tqdm import tqdm

def get_embedding(sequence, max_length=512):
    inputs = tokenizer(
        sequence,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    embedding = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
    return embedding

embeddings = []
for seq in tqdm(sequences):
    emb = get_embedding(seq)
    embeddings.append(emb)

embeddings = np.array(embeddings)
print(f"Размер матрицы: {embeddings.shape}")

100%|██████████| 9212/9212 [02:56<00:00, 52.32it/s]

Размер матрицы: (9212, 480)


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import pickle

# Делим на train/test
X_train, X_test, y_train, y_test = train_test_split(
    embeddings, labels, test_size=0.2, random_state=42
)

# Обучаем
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

# Оцениваем
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

# Сохраняем
with open("plastic_classifier.pkl", "wb") as f:
    pickle.dump(clf, f)

print("Модель сохранена в plastic_classifier.pkl")

              precision    recall  f1-score   support

           0       0.99      1.00      1.00       995
           1       1.00      0.99      1.00       848

    accuracy                           1.00      1843
   macro avg       1.00      1.00      1.00      1843
weighted avg       1.00      1.00      1.00      1843

Модель сохранена в plastic_classifier.pkl


In [7]:
def predict_plastic(sequence):
    emb = get_embedding(sequence)
    proba = clf.predict_proba([emb])[0][1]
    label = clf.predict([emb])[0]
    result = "plastic" if label == 1 else "not plastic"
    print(f"{result} ({proba:.2%})")
    return label, proba

predict_plastic(sequences[0])

plastic (99.97%)


(np.int64(1), np.float64(0.9997411926029702))

In [8]:
script = '''
import torch
import numpy as np
import pickle
from transformers import AutoTokenizer, AutoModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t12_35M_UR50D")
model = AutoModel.from_pretrained("facebook/esm2_t12_35M_UR50D")
model = model.to(device)
model.eval()

with open("plastic_classifier.pkl", "rb") as f:
    clf = pickle.load(f)

def get_embedding(sequence, max_length=512):
    inputs = tokenizer(sequence, return_tensors="pt", truncation=True, max_length=max_length)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()

def predict(sequence):
    emb = get_embedding(sequence)
    proba = clf.predict_proba([emb])[0][1]
    label = clf.predict([emb])[0]
    result = "plastic" if label == 1 else "not plastic"
    print(f"{result} ({proba:.2%})")
    return label, proba

if __name__ == "__main__":
    seq = input("sequence: ")
    predict(seq)
'''

with open("predict.py", "w") as f:
    f.write(script)

print("predict.py сохранён")

predict.py сохранён
